In [0]:
%python
from pyspark.sql.functions import col, sum as _sum, count, avg

catalog = "`prism-sentinel-stream`"
silver_table = f"{catalog}.prism_silver.transactions_refined"
gold_metrics_table = f"{catalog}.prism_gold.daily_risk_summary"

# Aggregate by Country and Risk Level
df_gold = spark.read.table(silver_table) \
    .groupBy("country", "risk_flag") \
    .agg(
        count("transaction_id").alias("txn_count"),
        _sum("amount").alias("total_value"),
        avg("similarity_score").alias("avg_similarity")
    )

# Save to Gold for Dashboarding
df_gold.write.format("delta").mode("overwrite").saveAsTable(gold_metrics_table)

print(f"✅ Gold metrics ready at {gold_metrics_table}")
